# 13 - 分布式训练 (AI Infra 视角)

本节从 **工程实现** 角度理解多 GPU 分布式训练：
- torchrun 启动机制
- DDP (分布式数据并行) 原理
- NCCL 通信原语
- 分布式数据加载
- 分布式优化器 (DistAdamW / DistMuon)
- ZeRO 优化策略

> 参考 nanochat/nanochat/common.py, adamw.py, muon.py, dataloader.py

In [ ]:
import torch
import torch.distributed as dist
import torch.nn as nn
import os
import math

## 1. 核心概念 (30秒版)

```
单卡训练:  1 个进程, 1 个 GPU, 看全部数据

DDP 训练:  N 个进程, N 个 GPU, 每个看 1/N 的数据
           每个进程独立 forward/backward
           梯度通过 NCCL 通信求平均
           参数更新后所有 GPU 保持一致

启动方式:
  torchrun --nproc-per-node=8 train.py
  → 启动 8 个 Python 进程
  → 自动设置 RANK, LOCAL_RANK, WORLD_SIZE 环境变量
  → 每个进程绑定一个 GPU
```

## 2. torchrun 做了什么

```bash
torchrun --nproc-per-node=8 scripts/base_train.py
```

等效于启动 8 个独立进程，每个进程的环境变量不同：

```
进程 0: RANK=0, LOCAL_RANK=0, WORLD_SIZE=8 → GPU 0
进程 1: RANK=1, LOCAL_RANK=1, WORLD_SIZE=8 → GPU 1
...  
进程 7: RANK=7, LOCAL_RANK=7, WORLD_SIZE=8 → GPU 7
```

多节点时的区别：

```bash
# 节点 A (2 GPU)
torchrun --nproc-per-node=2 --nnodes=2 --node-rank=0 \
         --master-addr=10.0.0.1 --master-port=29500 train.py

# 节点 B (2 GPU)
torchrun --nproc-per-node=2 --nnodes=2 --node-rank=1 \
         --master-addr=10.0.0.1 --master-port=29500 train.py

节点 A 进程 0: RANK=0, LOCAL_RANK=0  → 节点A GPU 0
节点 A 进程 1: RANK=1, LOCAL_RANK=1  → 节点A GPU 1
节点 B 进程 2: RANK=2, LOCAL_RANK=0  → 节点B GPU 0
节点 B 进程 3: RANK=3, LOCAL_RANK=1  → 节点B GPU 1
```

- `RANK`: 全局唯一编号
- `LOCAL_RANK`: 节点内编号（用于 `torch.cuda.set_device`）
- `WORLD_SIZE`: 总进程数

## 3. 进程组初始化

nanochat 的 `compute_init()` 做的事情：

```python
# nanochat/common.py
def compute_init(device_type="cuda"):
    # 1. 读取环境变量
    ddp_rank = int(os.environ["RANK"])
    ddp_local_rank = int(os.environ["LOCAL_RANK"])
    ddp_world_size = int(os.environ["WORLD_SIZE"])
    
    # 2. 绑定 GPU
    device = torch.device("cuda", ddp_local_rank)
    torch.cuda.set_device(device)
    
    # 3. 初始化进程组 (所有进程在此会合)
    dist.init_process_group(backend="nccl", device_id=device)
    
    # 4. 同步屏障 (确保所有进程都完成初始化)
    dist.barrier()
```

In [ ]:
# 模拟 nanochat 的 DDP 检测逻辑

def is_ddp_requested() -> bool:
    """检查是否由 torchrun 启动"""
    return all(k in os.environ for k in ("RANK", "LOCAL_RANK", "WORLD_SIZE"))

def get_dist_info():
    """获取分布式训练信息"""
    if is_ddp_requested():
        ddp_rank = int(os.environ["RANK"])
        ddp_local_rank = int(os.environ["LOCAL_RANK"])
        ddp_world_size = int(os.environ["WORLD_SIZE"])
        return True, ddp_rank, ddp_local_rank, ddp_world_size
    else:
        return False, 0, 0, 1  # 单卡默认值

ddp, rank, local_rank, world_size = get_dist_info()
print(f"DDP requested: {ddp}")
print(f"Rank: {rank}, Local rank: {local_rank}, World size: {world_size}")
print(f"\n当你用 torchrun 启动时, DDP requested 为 True")
print(f"直接用 python 运行时, DDP requested 为 False → 单卡模式")

## 4. NCCL 通信原语

NCCL (NVIDIA Collective Communications Library) 是 GPU 间通信的底层库。

```
类比：NCCL 之于 GPU 通信，就像 TCP/IP 之于网络通信
      应用层不直接调 NCCL，而是通过 torch.distributed 调用
```

分布式训练用到的核心通信操作：

| 操作 | 作用 | 场景 |
|------|------|------|
| **all_reduce** | 所有 GPU 求和/平均，结果广播到所有 GPU | 梯度同步、loss 汇总 |
| **reduce_scatter** | 求和/平均后，结果分片到各 GPU | ZeRO 优化器梯度分片 |
| **all_gather** | 每个 GPU 的数据拼接，结果广播到所有 GPU | ZeRO 参数同步 |
| **barrier** | 所有进程等待彼此 | 同步检查点 |

In [ ]:
# 用 CPU tensor 模拟 NCCL 通信原语
# (真实场景用 GPU tensor + dist.all_reduce 等)

def simulate_all_reduce(tensors, op="avg"):
    """模拟 all_reduce: 所有 GPU 的 tensor 求平均，结果广播到所有 GPU"""
    total = sum(t for t in tensors)
    if op == "avg":
        total = total / len(tensors)
    return [total.clone() for _ in tensors]

def simulate_reduce_scatter(tensors, op="avg"):
    """模拟 reduce_scatter: 求平均后，每个 GPU 只拿结果的一部分"""
    total = sum(t for t in tensors)
    if op == "avg":
        total = total / len(tensors)
    chunk_size = total.shape[0] // len(tensors)
    return [total[i*chunk_size:(i+1)*chunk_size].clone() for i in range(len(tensors))]

def simulate_all_gather(chunks):
    """模拟 all_gather: 每个 GPU 贡献一部分，拼成完整结果广播给所有 GPU"""
    full = torch.cat(chunks)
    return [full.clone() for _ in chunks]

# === 演示 all_reduce ===
print("=== all_reduce (AVG) ===")
print("场景: 4 个 GPU 各自算了梯度，需要求平均")
grads = [torch.tensor([1.0, 2.0, 3.0, 4.0]) * (i + 1) for i in range(4)]
for i, g in enumerate(grads):
    print(f"  GPU {i} 梯度: {g.tolist()}")
result = simulate_all_reduce(grads, op="avg")
print(f"  all_reduce 后每个 GPU 都得到: {result[0].tolist()}")

In [ ]:
# === 演示 reduce_scatter ===
print("=== reduce_scatter (AVG) ===")
print("场景: 4 个 GPU 求平均后，每个 GPU 只拿 1/4 的结果")
grads = [torch.tensor([1.0, 2.0, 3.0, 4.0]) * (i + 1) for i in range(4)]
for i, g in enumerate(grads):
    print(f"  GPU {i} 梯度: {g.tolist()}")
result = simulate_reduce_scatter(grads, op="avg")
for i, r in enumerate(result):
    print(f"  GPU {i} 拿到: {r.tolist()}")

print()

# === 演示 all_gather ===
print("=== all_gather ===")
print("场景: 每个 GPU 更新了自己的一部分参数，拼回完整参数")
chunks = [torch.tensor([10.0 + i]) for i in range(4)]
for i, c in enumerate(chunks):
    print(f"  GPU {i} 的参数片段: {c.tolist()}")
result = simulate_all_gather(chunks)
print(f"  all_gather 后每个 GPU 都得到: {result[0].tolist()}")

### 通信拓扑

```
同一节点内 (8×H100):
  NVLink/NVSwitch: ~900 GB/s 双向带宽
  所有 GPU 全互联, 通信几乎无瓶颈

跨节点:
  InfiniBand RDMA: ~400 GB/s
  带宽只有节点内的一半, 是通信瓶颈所在

消费级 (你的 RTX 5070 Ti):
  PCIe 5.0 x16: ~64 GB/s
  单卡无需通信, 多卡用 PCIe 比 NVLink 慢很多
```

## 5. DDP 的梯度同步

经典 DDP 的工作方式：

```
GPU 0: forward → backward → ┐
GPU 1: forward → backward → ├─ all_reduce(AVG) → 每个 GPU 得到相同的平均梯度
GPU 2: forward → backward → ┤                   → optimizer.step()
GPU 3: forward → backward → ┘                   → 参数保持一致
```

PyTorch 官方 DDP 包装器：
```python
model = torch.nn.parallel.DistributedDataParallel(model, device_ids=[local_rank])
```

**但 nanochat 没有用 DDP 包装器！** 它在优化器层面自己做通信（DistAdamW / DistMuon），更灵活也更高效。

## 6. 分布式数据加载

DDP 的核心要求：**每个 GPU 看不同的数据**，不然梯度平均没意义。

nanochat 的做法：按 `rank` 交错读取 parquet 文件的 row groups。

```python
# nanochat/dataloader.py 的核心逻辑
rg_idx = ddp_rank                    # 从自己的 rank 开始
while rg_idx < pf.num_row_groups:
    rg = pf.read_row_group(rg_idx)   # 读属于自己的 row group
    rg_idx += ddp_world_size          # 跳过其他 rank 的数据
```

In [ ]:
# 可视化数据分片策略

num_row_groups = 16
world_size = 4

print(f"Parquet 文件有 {num_row_groups} 个 row groups, {world_size} 个 GPU")
print()

for rank in range(world_size):
    rg_indices = list(range(rank, num_row_groups, world_size))
    print(f"  GPU {rank}: row groups {rg_indices}")

print()
print("每个 GPU 读取的数据完全不重叠 → 等效 N 倍数据吞吐")
print("all_reduce 平均梯度后 → 等效于在全部数据上训练")

## 7. ZeRO 优化 — 为什么 nanochat 不用 DDP 包装器

经典 DDP 的问题：
```
每个 GPU 都存完整的:
  - 模型参数        (2 bytes/param, bf16)
  - 梯度            (2 bytes/param)
  - 优化器状态       (8 bytes/param, Adam 的 m+v fp32)
  
总计: ~12 bytes/param × N 个 GPU = 大量冗余
```

ZeRO (Zero Redundancy Optimizer) 的思路：
```
ZeRO-1: 优化器状态分片 (每个 GPU 只存 1/N)
ZeRO-2: + 梯度也分片
ZeRO-3: + 参数也分片 (每个 GPU 只存 1/N 的参数)
```

nanochat 的 DistAdamW 使用 **ZeRO-2 风格**：
```
1. reduce_scatter: 梯度求平均并分片 → 每个 GPU 只拿 1/N
2. 每个 GPU 用自己那份梯度更新 1/N 的参数
3. all_gather: 把更新后的参数拼回完整版
```

In [ ]:
# 对比经典 DDP vs ZeRO-2 的显存占用

def memory_comparison(num_params_b, world_size):
    """计算不同策略下每个 GPU 的显存占用 (GB)"""
    params_bytes = num_params_b * 1e9 * 2  # bf16
    grads_bytes = num_params_b * 1e9 * 2   # bf16 
    opt_bytes = num_params_b * 1e9 * 8     # Adam m+v in fp32
    
    # 经典 DDP: 每个 GPU 存完整的一切
    ddp_per_gpu = (params_bytes + grads_bytes + opt_bytes) / 1e9
    
    # ZeRO-2: 优化器状态和梯度分片
    zero2_per_gpu = (params_bytes + (grads_bytes + opt_bytes) / world_size) / 1e9
    
    # ZeRO-3: 全部分片
    zero3_per_gpu = (params_bytes + grads_bytes + opt_bytes) / world_size / 1e9
    
    print(f"模型: {num_params_b}B 参数, {world_size} 个 GPU")
    print(f"  经典 DDP:  {ddp_per_gpu:.1f} GB/GPU")
    print(f"  ZeRO-2:   {zero2_per_gpu:.1f} GB/GPU  (省 {(1-zero2_per_gpu/ddp_per_gpu)*100:.0f}%)")
    print(f"  ZeRO-3:   {zero3_per_gpu:.1f} GB/GPU  (省 {(1-zero3_per_gpu/ddp_per_gpu)*100:.0f}%)")
    print()

# nanochat 的配置: 2.2B 参数, 8×H100
memory_comparison(2.2, 8)

# 更大模型
memory_comparison(7, 8)

## 8. DistAdamW — ZeRO-2 风格的分布式 AdamW

```python
# nanochat/adamw.py 核心逻辑 (简化)

class DistAdamW(Optimizer):
    def step(self):
        rank = dist.get_rank()
        world_size = dist.get_world_size()
        
        # ① reduce_scatter: 梯度求平均并分片
        for p in params:
            rank_size = p.shape[0] // world_size
            grad_slice = empty(rank_size)  # 只接收 1/N
            dist.reduce_scatter_tensor(grad_slice, p.grad, op=AVG)
        
        # ② 每个 rank 只更新自己负责的 1/N 参数
        for p in params:
            p_slice = p[rank*rank_size : (rank+1)*rank_size]
            # 标准 AdamW 更新 (只在 p_slice 上)
            exp_avg.mul_(beta1).add_(grad_slice, alpha=1-beta1)
            exp_avg_sq.mul_(beta2).addcmul_(grad_slice, grad_slice, value=1-beta2)
            p_slice.add_(update, alpha=-lr)
        
        # ③ all_gather: 把各 rank 更新的参数拼回完整版
        for p in params:
            dist.all_gather_into_tensor(p, p_slice)
```

关键点：`p.shape[0]` 必须能被 `world_size` 整除（参数按第一个维度切分）

In [ ]:
# 模拟 DistAdamW 的分片逻辑

world_size = 4
param_shape = (512, 256)  # embedding 或 lm_head 的一个参数

print(f"参数形状: {param_shape}")
print(f"World size: {world_size}")
print(f"第一维 {param_shape[0]} / {world_size} = {param_shape[0]//world_size}")
print()

# 每个 rank 负责的切片
rank_size = param_shape[0] // world_size
for rank in range(world_size):
    start = rank * rank_size
    end = (rank + 1) * rank_size
    print(f"  Rank {rank}: p[{start}:{end}] (shape {rank_size}×{param_shape[1]})")
    print(f"           优化器状态: exp_avg ({rank_size}×{param_shape[1]}), exp_avg_sq ({rank_size}×{param_shape[1]})")

full_opt_mem = param_shape[0] * param_shape[1] * 8  # m + v in fp32
shard_opt_mem = rank_size * param_shape[1] * 8
print(f"\n优化器显存: {full_opt_mem/1e6:.1f}MB (全量) → {shard_opt_mem/1e6:.1f}MB (分片, 省 {(1-1/world_size)*100:.0f}%)")

## 9. DistMuon — Block-Cyclic 分配

DistMuon 和 DistAdamW 不同，它不是按参数内部切片，而是按 **参数个数** 分配：

```
8 个参数, 4 个 GPU:
  GPU 0 负责: param[0], param[4]
  GPU 1 负责: param[1], param[5]
  GPU 2 负责: param[2], param[6]
  GPU 3 负责: param[3], param[7]

这叫 Block-Cyclic 分配 (循环分配)
```

流程：
```
① reduce_scatter: 每组 world_size 个参数的梯度求平均
   → 每个 rank 只拿到自己负责的那个参数的平均梯度

② owner rank 做 Muon 更新:
   momentum → Nesterov → Newton-Schulz 正交化 → 更新参数

③ all_gather: 把更新后的参数广播给所有 rank
```

优势：每个参数的 Muon 动量缓冲只在 owner rank 存储，显存省 N 倍。

In [ ]:
# 模拟 DistMuon 的 block-cyclic 分配

num_params = 12
world_size = 4

print(f"{num_params} 个 Muon 参数, {world_size} 个 GPU")
print()

# Block-cyclic 分配
for rank in range(world_size):
    owned = [i for i in range(rank, num_params, world_size)]
    print(f"  GPU {rank} 负责: param{owned}")
    print(f"         只在这些参数上维护 momentum buffer")

print()
print("通信流程 (每 world_size 个参数为一组):")
for base_i in range(0, num_params, world_size):
    group = list(range(base_i, min(base_i + world_size, num_params)))
    print(f"  组 {group}:")
    print(f"    reduce_scatter → 每个 GPU 拿到自己负责的参数的平均梯度")
    print(f"    各 GPU 独立做 Muon 更新")
    print(f"    all_gather → 更新后的参数广播给所有 GPU")

## 10. 异步通信 — 计算与通信重叠

nanochat 大量使用 `async_op=True` 实现通信和计算的重叠：

```python
# nanochat 的模式:
# 1. 启动异步通信
future = dist.reduce_scatter(output, input, async_op=True).get_future()

# 2. 做其他事情 (比如启动下一组的通信)
...

# 3. 需要结果时再等待
future.wait()
```

```
同步通信:                         异步通信:
GPU:   [计算][等通信][计算]        GPU:   [计算][计算][计算]
NCCL:        [通信]               NCCL:  [通信][通信]  ← 重叠
```

In [ ]:
# DistMuon step 的完整流程

print("DistMuon.step() 的执行流程:")
print()
print("Phase 1: 启动所有 reduce_scatter (异步)")
print("  for each group of world_size params:")
print("    future = dist.reduce_scatter(grad, async_op=True)")
print("    futures.append(future)")
print()
print("Phase 2: 逐组等待 + 更新 + 启动 all_gather (异步)")
print("  for each group:")
print("    future.wait()                    # 等这组的 reduce_scatter 完成")
print("    if I own this param:")
print("      buf.lerp_(g, 1-momentum)       # 动量更新")
print("      g = g.lerp_(buf, momentum)     # Nesterov")
print("      g = newton_schulz(g)           # 正交化")
print("      p.add_(g, alpha=-lr*scale)     # 更新参数")
print("    future = dist.all_gather(p, async_op=True)")
print()
print("Phase 3: 等待所有 all_gather 完成")
print("  torch.futures.collect_all(futures).wait()")

## 11. 分布式评估

评估也需要分布式协作，nanochat 的做法：

### 验证 Loss (loss_eval.py)
```python
# 每个 GPU 在不同数据上算 loss，最后 all_reduce 汇总
if world_size > 1:
    dist.all_reduce(total_nats, op=dist.ReduceOp.SUM)
    dist.all_reduce(total_bytes, op=dist.ReduceOp.SUM)
bpb = total_nats / total_bytes
```

### CORE 评估 (core_eval.py)
```python
# 按 rank 分配评估样本
for idx in range(rank, len(data), world_size):  # 交错分配
    is_correct = evaluate_example(idx, ...)

# 汇总正确数
dist.barrier()
dist.all_reduce(correct, op=dist.ReduceOp.SUM)
```

同样的模式：**分片计算 → 通信汇总**

## 12. 完整训练步的通信模式

```
梯度累积 (无通信):
  for micro_step in range(grad_accum_steps):
      loss = model(x, y)       # 各 GPU 独立前向
      loss.backward()          # 各 GPU 独立反向, 梯度累加
      x, y = next(loader)      # 各 GPU 读不同数据

优化器 (有通信):
  DistAdamW.step():            
    reduce_scatter(grad)       ← 通信: 梯度求平均并分片
    更新 1/N 的参数              # 各 rank 独立计算
    all_gather(param)          ← 通信: 参数同步
    
  DistMuon.step():
    reduce_scatter(grad)       ← 通信: 梯度求平均
    Muon 更新自己负责的参数       # owner rank 计算
    all_gather(param)          ← 通信: 参数同步

同步:
  synchronize()                # 确保所有 GPU 完成
```

注意：nanochat 没有在 forward/backward 中做通信（不像 PyTorch DDP 在 backward 时自动 all_reduce），所有通信集中在 optimizer.step() 中。

## 13. print0 和 master_process

N 个进程跑同一份代码，日志只需要打印一次：

```python
# nanochat/common.py
def print0(s="", **kwargs):
    ddp_rank = int(os.environ.get('RANK', 0))
    if ddp_rank == 0:      # 只有 rank 0 打印
        print(s, **kwargs)

# base_train.py
master_process = ddp_rank == 0
if master_process:         # 采样、保存检查点只在 rank 0 做
    engine.generate_batch(...)
    save_checkpoint(...)
```

如果不这么做，8 个 GPU 会打印 8 遍一模一样的日志。

In [ ]:
# 模拟 print0 的行为

def print0(s="", rank=0, **kwargs):
    if rank == 0:
        print(s, **kwargs)

print("没有 print0 的日志 (8 个 GPU):")
for rank in range(8):
    print(f"  [Rank {rank}] Loss: 4.2000")

print()
print("有 print0 的日志:")
for rank in range(8):
    print0(f"  Loss: 4.2000", rank=rank)

## 14. 单卡到多卡：代码几乎不变

nanochat 的设计让单卡和多卡用同一份代码：

```python
# gpt.py: setup_optimizers
ddp, rank, local_rank, world_size = get_dist_info()

# 自动选择分布式 or 单机优化器
AdamWFactory = DistAdamW if ddp else partial(torch.optim.AdamW, fused=True)
MuonFactory = DistMuon if ddp else Muon
```

| 启动方式 | 优化器 | 通信 |
|---------|--------|------|
| `python train.py` | AdamW + Muon | 无 |
| `torchrun --nproc-per-node=8 train.py` | DistAdamW + DistMuon | NCCL |

唯一的区别就是启动命令，训练脚本不需要改一行代码。

## 15. 面试常见问题

### Q1: DDP 和 DP (DataParallel) 的区别？

**答**:
- DP: 单进程多线程，受 GIL 限制，所有通信过 CPU → 慢
- DDP: 多进程，每个进程独立，GPU 直接通过 NCCL 通信 → 快
- DP 已经基本弃用，现在都用 DDP

---

### Q2: all_reduce vs reduce_scatter + all_gather？

**答**:
- `all_reduce`: 简单，每个 GPU 都得到完整的平均梯度。经典 DDP 用这个。
- `reduce_scatter` + `all_gather`: ZeRO 用这个。reduce_scatter 后每个 GPU 只拿 1/N，省显存。更新完再 all_gather 拼回来。
- 通信量相同，但后者能省优化器状态的显存。

---

### Q3: 为什么 nanochat 不用 PyTorch 的 DDP 包装器？

**答**:
- PyTorch DDP 在 backward 期间自动 all_reduce 梯度
- nanochat 想用 ZeRO-2 风格的分片优化器 (reduce_scatter + all_gather)
- 自己控制通信更灵活，可以做异步重叠
- 避免了 DDP 的开销 (bucket 管理、hook 注册等)

---

### Q4: 梯度累积时需要通信吗？

**答**:
- 不需要。梯度累积期间每个 GPU 独立做 forward/backward
- 只在累积完成后的 optimizer.step() 中做通信
- 如果用 PyTorch DDP 包装器，需要 `model.no_sync()` 来禁止中间步骤的通信

---

### Q5: dist.barrier() 什么时候用？

**答**:
- 初始化完成后同步 (确保所有进程都准备好)
- 保存检查点前 (确保所有 rank 训练到同一步)
- 评估汇总前 (确保所有 rank 都算完了)
- 尽量少用，因为 barrier 会让快的进程等慢的

---

### Q6: 如果一个进程挂了怎么办？

**答**:
- 其他进程在下一次通信操作时会超时 (NCCL timeout)
- 所有进程都会崩溃
- 需要从最近的检查点重启全部进程
- 这就是为什么 save_every 很重要

## 16. 总结速查表

| 主题 | 要点 |
|------|------|
| **启动方式** | `torchrun --nproc-per-node=N` 自动设置 RANK/LOCAL_RANK/WORLD_SIZE |
| **通信后端** | NCCL (GPU), gloo (CPU) |
| **数据分片** | 每个 rank 读不同的 row groups (交错分配) |
| **经典 DDP** | backward 时 all_reduce 梯度 |
| **nanochat** | optimizer.step() 中 reduce_scatter + all_gather (ZeRO-2) |
| **DistAdamW** | 参数按第一维切片，每个 rank 更新 1/N |
| **DistMuon** | 参数按个数 block-cyclic 分配，owner rank 做 Muon 更新 |
| **异步通信** | async_op=True，计算与通信重叠 |
| **日志** | 只在 rank 0 打印 (print0) |
| **容错** | 一个进程挂 → 全部挂，从检查点恢复 |

### 通信操作速查

```
all_reduce:       [A,B,C,D] → [A+B+C+D] (每个 GPU 都有完整结果)
reduce_scatter:   [A,B,C,D] → GPU0=[A+..], GPU1=[B+..], ... (分片结果)
all_gather:       GPU0=[a], GPU1=[b], ... → [a,b,c,d] (每个 GPU 都有完整结果)
barrier:          所有进程互相等待
```